In [1]:
import os, random
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import plotly.io as pio
pio.renderers.default='notebook'

import tensorflow as tf
from tensorboard import notebook
%load_ext tensorboard

In [2]:
tf.__version__

'2.1.0'

## Загрузка данных

In [3]:
train_df = pd.read_csv('datas/project/fashion-mnist_train.csv')
test_df = pd.read_csv('datas/project/fashion-mnist_test.csv')

In [4]:
# разбитие по переменным 
y_train, x_train = train_df.iloc[:,0], train_df.iloc[:,1:]
y_test, x_test = test_df.iloc[:,0], test_df.iloc[:,1:]
# нормализация признаков
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)
# представление ответов в One-Hot Encoding
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)

In [5]:
y_train.shape, x_train.shape

((60000, 10), (60000, 784))

In [6]:
random.seed(42)

## Логистическая регрессия

Для решения задачи классификации предлагается начать с использования логистической регрессии. В данном случае, количество признаков равно 28x28=784, так же мы имеем 60000 объектов в тренировочной выборке. Поэтому рекомендуется использовать tensorflow или keras для выполнения этого задания. Используйте стохастический градиентный спуск (stochastic gradient descent) в качестве алгоритма оптимизации.

По своей сути, логистическая регрессия может быть реализована как нейронная сеть без скрытых слоев. В выходном слое содержится количество нейронов, равное количеству классов. В качестве функции активации выходного слоя следует использовать softmax.

Обучите логистическую регрессию на тренировочной выборке и оцените качество на тестовой выборке используя метрику accuracy. Постройте график качества модели на валидационной выборке от количества эпох. Для этого вы можете использовать утилиту tensorboard.
<ul>
    <li><a href="https://www.tensorflow.org/guide/summaries_and_tensorboard" target="_blank">tensorboard в tensorflow</a></li>
    <li><a href="https://keras.io/callbacks/#tensorboard" target="_blank">tensorboard в keras</a></li>
</ul>

In [7]:
log_reg_path = 'datas/project/logs/log_reg' # папка для хранения данных для tensorflow

# функция построения модели логистической регрессии
def log_reg(x_train, y_train, epochs, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(10, activation='softmax', input_shape=(784,)),
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='sgd',
        metrics=['accuracy'],
    )

    logs= [tf.keras.callbacks.TensorBoard(
        log_dir = log_reg_path,
        write_graph=True,
        write_images=True,
        histogram_freq=1,
        profile_batch=100000000,
    )]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_split=0.2,
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs,
        use_multiprocessing=True,
    )
    return model

In [8]:
model_log_reg = log_reg(x_train, y_train, epochs=10, batch_size=1000)
_, accuracy_log_reg = model_log_reg.evaluate(x_test, y_test)

Train on 48000 samples, validate on 12000 samples
Epoch 1/10
48000/48000 [==============================] - 1s 29us/sample - loss: 2.2961 - accuracy: 0.1621 - val_loss: 2.2927 - val_accuracy: 0.1764
Epoch 2/10
48000/48000 [==============================] - 1s 25us/sample - loss: 2.2882 - accuracy: 0.2050 - val_loss: 2.2849 - val_accuracy: 0.2202
Epoch 3/10
48000/48000 [==============================] - 1s 22us/sample - loss: 2.2804 - accuracy: 0.2404 - val_loss: 2.2771 - val_accuracy: 0.2524
Epoch 4/10
48000/48000 [==============================] - 1s 16us/sample - loss: 2.2727 - accuracy: 0.2705 - val_loss: 2.2695 - val_accuracy: 0.2806
Epoch 5/10
48000/48000 [==============================] - 1s 15us/sample - loss: 2.2651 - accuracy: 0.2966 - val_loss: 2.2619 - val_accuracy: 0.3046
Epoch 6/10
48000/48000 [==============================] - 1s 15us/sample - loss: 2.2575 - accuracy: 0.3184 - val_loss: 2.2543 - val_accuracy: 0.3273
Epoch 7/10
48000/48000 [==============================] 

In [9]:
pd.DataFrame(model_log_reg.history.history) # фрейм всех метрик качества при обучении модели

,loss,accuracy,val_loss,val_accuracy
0,2.296086,0.162083,2.292735,0.176417
1,2.288209,0.205021,2.284889,0.220167
2,2.280420,0.240354,2.277131,0.252417
3,2.272716,0.270458,2.269456,0.280583
4,2.265091,0.296646,2.261860,0.304583
5,2.257539,0.318437,2.254339,0.327333
6,2.250058,0.338062,2.246888,0.342333
7,2.242642,0.353125,2.239505,0.357750
8,2.235293,0.367417,2.232188,0.370250
9,2.228005,0.379354,2.224932,0.381333


In [23]:
print('Точность модели логистической регрессии на валидационной выборке - {:.2f} %'.format(accuracy_log_reg*100))

accuracy = model_log_reg.history.history['accuracy']

log_reg_graph = go.Scatter(x=np.arange(len(accuracy)),
                           y=accuracy*100,
                           name='Логистическая регрессия',
)

layout = {'title': {'text': 'График качества модели логистической регрессии', # Заголовок графика
                    'x':0.5, # позиционирование title по центру
                   },
          'xaxis_title': 'Колличество эпох', # подпись Ох 
          'yaxis_title': 'Точность', # подпись Оу
         }

go.Figure(data=log_reg_graph,layout=layout).show()

Точность модели логистической регрессии на валидационной выборке - 38.11 %


## Полносвязная нейронная сеть

Далее, попробуйте реализовать полносвязную нейронную сеть с несколькими скрытыми слоями. Обучите модель и посчитайте качество на тестовой выборке. Как оно изменилось в сравнении с логистической регрессией? Как вы можете объяснить этот результат?

In [12]:
fcnn_path = 'datas/project/logs/fcnn'

# функция построения модели полносвязной нейронной сети
def fcnn(x_train, y_train, epochs, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(784, activation='relu', input_shape=(784,)),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='sgd',
        metrics=['accuracy'],
    )

    logs= [tf.keras.callbacks.TensorBoard(
        log_dir = fcnn_path,
        write_graph=True,
        write_images=True,
        histogram_freq=1,
        profile_batch=100000000,
    )]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_split=0.2,
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs,
        use_multiprocessing=True,
    )
    return model

In [13]:
model_fcnn = fcnn(x_train, y_train, epochs=10, batch_size=1000)
_, accuracy_fcnn = model_fcnn.evaluate(x_test, y_test)

Train on 48000 samples, validate on 12000 samples
Epoch 1/10
48000/48000 [==============================] - 5s 110us/sample - loss: 2.2957 - accuracy: 0.1256 - val_loss: 2.2882 - val_accuracy: 0.1764
Epoch 2/10
48000/48000 [==============================] - 4s 90us/sample - loss: 2.2814 - accuracy: 0.2233 - val_loss: 2.2741 - val_accuracy: 0.2765
Epoch 3/10
48000/48000 [==============================] - 5s 96us/sample - loss: 2.2675 - accuracy: 0.3374 - val_loss: 2.2603 - val_accuracy: 0.3842
Epoch 4/10
48000/48000 [==============================] - 4s 90us/sample - loss: 2.2540 - accuracy: 0.4249 - val_loss: 2.2469 - val_accuracy: 0.4553
Epoch 5/10
48000/48000 [==============================] - 5s 94us/sample - loss: 2.2406 - accuracy: 0.4824 - val_loss: 2.2337 - val_accuracy: 0.5108
Epoch 6/10
48000/48000 [==============================] - 5s 94us/sample - loss: 2.2275 - accuracy: 0.5371 - val_loss: 2.2206 - val_accuracy: 0.5551
Epoch 7/10
48000/48000 [==============================]

In [24]:
print('Точность модели полносвязной нейронной сети на валидационной выборке - {:.2f} %'.format(accuracy_fcnn*100))

accuracy = model_fcnn.history.history['accuracy']

fcnn_graph = go.Scatter(x=np.arange(len(accuracy)),
                        y=accuracy*100,
                        name='Полносвязная нейронная сеть',
)

layout = {'title': {'text': 'График качества модели полносвязной нейронной сети', # Заголовок графика
                    'x':0.5, # позиционирование title по центру
                   },
          'xaxis_title': 'Колличество эпох', # подпись Ох 
          'yaxis_title': 'Точность', # подпись Оу
         }

go.Figure(data=fcnn_graph,layout=layout).show()

Точность модели полносвязной нейронной сети на валидационной выборке - 60.15 %


## Сверточная нейронная сеть

После этого вам предлагается реализовать сверточную нейронную сеть. В данном случае лучше использовать готовые слои, которые предоставляют keras или tensorflow.

Начните с модели с несколькими сверточными слоями. Так же рекомендуется использовать слои суб-дискретизации, например Max Pooling слои. Они понижают размерность сходных данных и выделяют наиболее важные признаки из данных. Посчитайте качество получившейся модели на тестовой выборке. Сравните полученные результаты с результатами полносвязной нейронной сети.

In [15]:
cnn1_path = 'datas/project/logs/cnn1/'

# функция построения модели сверточной нейронной сети
def cnn1(x_train, y_train, epochs, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Convolution2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Convolution2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='sgd',
        metrics=['accuracy'],
    )

    logs= [tf.keras.callbacks.TensorBoard(
        log_dir = cnn1_path,
        write_graph=True,
        write_images=True,
        histogram_freq=1,
        profile_batch=100000000,
    )]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_split=0.2,
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs,
        use_multiprocessing=True,
    )
    return model

In [16]:
model_cnn1 = cnn1(x_train.values.reshape(60000, 28, 28, 1), y_train, epochs=10, batch_size=1000)
_, accuracy_cnn1 = model_cnn1.evaluate(x_test.values.reshape(10000, 28, 28, 1), y_test)

Train on 48000 samples, validate on 12000 samples
Epoch 1/10
48000/48000 [==============================] - 40s 838us/sample - loss: 2.3004 - accuracy: 0.1123 - val_loss: 2.2988 - val_accuracy: 0.1152
Epoch 2/10
48000/48000 [==============================] - 38s 788us/sample - loss: 2.2980 - accuracy: 0.1226 - val_loss: 2.2973 - val_accuracy: 0.1214
Epoch 3/10
48000/48000 [==============================] - 39s 805us/sample - loss: 2.2968 - accuracy: 0.1396 - val_loss: 2.2962 - val_accuracy: 0.1257
Epoch 4/10
48000/48000 [==============================] - 38s 785us/sample - loss: 2.2957 - accuracy: 0.1406 - val_loss: 2.2952 - val_accuracy: 0.1376
Epoch 5/10
48000/48000 [==============================] - 38s 786us/sample - loss: 2.2948 - accuracy: 0.1443 - val_loss: 2.2944 - val_accuracy: 0.1447
Epoch 6/10
48000/48000 [==============================] - 38s 798us/sample - loss: 2.2939 - accuracy: 0.1535 - val_loss: 2.2935 - val_accuracy: 0.1523
Epoch 7/10
48000/48000 [====================

In [17]:
print('Точность модели сверточной нейронной сети на валидационной выборке - {:.2f} %'.format(accuracy_cnn1*100))
accuracy = model_cnn1.history.history['accuracy']

cnn1_graph = go.Scatter(x=np.arange(len(accuracy)),
                        y=accuracy,
                        name='Сверточная нейронная сеть',
)

layout = {'title': {'text': 'График качества модели сверточной нейронной сети', # Заголовок графика
                    'x':0.5, # позиционирование title по центру
                   },
          'xaxis_title': 'Колличество эпох', # подпись Ох 
          'yaxis_title': 'Точность', # подпись Оу
         }

go.Figure(data=cnn1_graph,layout=layout).show()

Точность модели сверточной нейронной сети на валидационной выборке - 21.75 %


Далее, попробуйте увеличить количество слоев в вашей нейронной сети. Достаточно добавить несколько новых сверточных слоев. Проанализируете, как изменилось качество в этом случае.

В заключение, рекомендуется попробовать добавить Batch Normalization слои. Обычно они располагаются после сверточных слоев или слоев полносвязной нейронной сети. Обычно они улучшают качество модели, этим объясняется их популярность использования в современных архитектурах нейронных сетей. Однако, это требует проверки для конкретной модели и конкретного набора данных.

In [18]:
cnn2_path = 'datas/project/logs/cnn2/'
# функция построения модели полносвязной нейронной сети
def cnn2(x_train, y_train, epochs, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Convolution2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Convolution2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Convolution2D(128, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='sgd',
        metrics=['accuracy'],
    )

    logs= [tf.keras.callbacks.TensorBoard(
        log_dir = cnn2_path,
        write_graph=True,
        write_images=True,
        histogram_freq=1,
        profile_batch=100000000,
    )]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_split=0.2,
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs,
        use_multiprocessing=True,
    )
    return model

In [19]:
model_cnn2 = cnn2(x_train.values.reshape(60000, 28, 28, 1), y_train, epochs=10, batch_size=1000)
_, accuracy_cnn2 = model_cnn2.evaluate(x_test.values.reshape(10000, 28, 28, 1), y_test)

Train on 48000 samples, validate on 12000 samples
Epoch 1/10
48000/48000 [==============================] - 44s 914us/sample - loss: 2.1973 - accuracy: 0.3895 - val_loss: 2.2909 - val_accuracy: 0.1989
Epoch 2/10
48000/48000 [==============================] - 43s 896us/sample - loss: 1.9828 - accuracy: 0.4898 - val_loss: 2.2600 - val_accuracy: 0.2171
Epoch 3/10
48000/48000 [==============================] - 43s 898us/sample - loss: 1.6790 - accuracy: 0.5385 - val_loss: 2.1918 - val_accuracy: 0.4403
Epoch 4/10
48000/48000 [==============================] - 44s 911us/sample - loss: 1.3971 - accuracy: 0.5900 - val_loss: 2.0942 - val_accuracy: 0.4928
Epoch 5/10
48000/48000 [==============================] - 44s 908us/sample - loss: 1.1846 - accuracy: 0.6325 - val_loss: 1.9698 - val_accuracy: 0.5890
Epoch 6/10
48000/48000 [==============================] - 43s 899us/sample - loss: 1.0410 - accuracy: 0.6619 - val_loss: 1.8223 - val_accuracy: 0.6275
Epoch 7/10
48000/48000 [====================

In [25]:
print('Точность модели сверточной нейронной сети с большим кол-во скрытых слоев на валидационной выборке - {:.2f} %'.format(accuracy_cnn2*100))

accuracy = model_cnn2.history.history['accuracy']

cnn2_graph = go.Scatter(x=np.arange(len(accuracy)),
                        y=accuracy,
                        name='Сверточная нейронная сеть',
)

layout = {'title': {'text': 'График качества модели сверточной нейронной сети с большим кол-во скрытых слоев', # Заголовок графика
                    'x':0.5, # позиционирование title по центру
                   },
          'xaxis_title': 'Колличество эпох', # подпись Ох 
          'yaxis_title': 'Точность', # подпись Оу
         }

go.Figure(data=cnn2_graph,layout=layout).show()

Точность модели сверточной нейронной сети с большим кол-во скрытых слоев на валидационной выборке - 70.41 %


## Ответ

В качестве решения приложите архив, содержащий файл решения и все используемые для его работы файлы.Постройте график качества модели на валидационной выборке от количества эпох. Для этого вы можете использовать утилиту tensorboard.

In [40]:
# проверка данных с помощью tensorboard
common_path = 'datas/project/logs/'
os.makedirs(common_path, exist_ok=True)

%tensorboard --logdir=$common_path --host localhost

Reusing TensorBoard on port 6006 (pid 2536), started 2 days, 21:24:09 ago. (Use '!kill 2536' to kill it.)

In [39]:
print('Точность моделей, проверенная на тестовых данных:\n\tЛогистическая регрессия: {:.2f} %;\n\tПолносвязная нейронная сеть: {:.2f} %;\n\tСверточная нейронная сеть: {:.2f} %;\n\tСверточная нейронная сеть с большим кол-во скрытых слоев: {:.2f} %'.format(
    accuracy_log_reg*100, accuracy_fcnn*100, accuracy_cnn1*100, accuracy_cnn2*100))

Точность моделей, проверенная на тестовых данных:
	Логистическая регрессия: 38.11 %;
	Полносвязная нейронная сеть: 60.15 %;
	Сверточная нейронная сеть: 21.75 %;
	Сверточная нейронная сеть с большим кол-во скрытых слоев: 70.41 %


Каждая модель обучалась дольше предыдущей.<br> Самый лучший результат показала модель <b>cверточной нейронной сети с большим кол-вом скрытых слоев</b>, самый худший результат показала модель <b>cверточной нейронной сети c 2мя сверточными слоями</b>. Это доказывает, что самая <i>сложная модель не всегда приводит к лучшему результату</i>, но <i>при увеличении кол-ва слоев, качество модели увеличивается</i>.